# Fundamento 4 — De modelo a API en vivo con KServe

## Explicación

`model.pkl` es solo un archivo en disco: solo puede invocarse desde un script de Python. Este notebook reproduce lo que hace `inference/src/predictor.py` (el contrato de entrada/salida de la API) **sin** necesitar Docker ni Kubernetes, y opcionalmente prueba la API real si ya está corriendo.

## Por qué es importante

Antes de empaquetar el modelo en un contenedor y desplegarlo en Kubernetes, hay que validar el "contrato": qué datos recibe la API, en qué orden exacto espera las columnas el modelo, y qué estructura de respuesta debe devolver. Si el orden de las features no coincide con el usado en entrenamiento, el modelo no lanza un error — simplemente predice mal, en silencio. Detectar esto aquí, en un notebook, es mucho más barato que detectarlo ya en producción.

## Mapa del notebook (para no perderse)

Este notebook **no despliega nada**: valida el contrato que después implementará la API real. Los pasos de Docker/Kubernetes están en la guía [`04-despliegue-kserve.md`](04-despliegue-kserve.md).

| Sección | Qué pasa ahí | Qué obtienes |
|---|---|---|
| 0. Preparativos | Cargar `artifacts/model.pkl` del Fundamento 3 | `pipeline` listo para predecir |
| El contrato de la API | Definir el orden de features y la función que las deriva | `FEATURE_ORDER`, `to_model_input()` |
| Simular `predictor.py` | Recorrer una petición completa: payload → features → predicción → JSON | La respuesta que devolvería la API |
| Probar la API real | POST de verdad a `/predict`, si la tienes levantada | La misma respuesta, ahora vía HTTP |

### El recorrido de una petición

```text
Frontend (formulario)
   └─▶ JSON con 15 campos "crudos" (snake_case)
        └─▶ to_model_input(): añade 4 features derivadas  → 19 features
             └─▶ DataFrame de 1 fila, en el ORDEN exacto de entrenamiento
                  └─▶ pipeline.predict_proba() → [P(se queda), P(se va)]
                       └─▶ JSON de respuesta: prediction + probabilidades + nivel de riesgo
```

> **La idea central del fundamento:** desplegar un modelo es, sobre todo, **respetar un contrato de datos**. El código de la API es simple; lo que rompe los despliegues es que la entrada no llegue exactamente como el modelo la espera.

## Librerías que usamos

| Librería | Para qué la usamos aquí |
|---|---|
| `json` (estándar) | Construir y mostrar el payload/respuesta de la API en formato JSON |
| `urllib.request` / `urllib.error` (estándar) | Hacer una petición HTTP POST real a la API, sin depender de `requests` |
| `pathlib.Path` (estándar) | Rutas de archivo multiplataforma |
| `joblib` | Cargar el `model.pkl` guardado en el Fundamento 3 |
| `pandas` | Armar la fila de entrada (`DataFrame` de 1 fila) en el orden exacto que espera el modelo |

## Comandos / funciones clave de este notebook

| Comando | Qué hace |
|---|---|
| `joblib.load(ruta)` | Carga de disco un objeto serializado con `joblib.dump` (aquí, el pipeline entrenado) |
| `pipeline.predict_proba(row)` | Devuelve `[P(se queda), P(se va)]` para cada fila de entrada |
| `urllib.request.Request(...)` | Arma una petición HTTP (URL, cuerpo, headers, método) |
| `urllib.request.urlopen(req, timeout=...)` | Envía la petición y espera la respuesta, con un límite de tiempo |
| `json.dumps(dict, indent=2)` | Convierte un diccionario de Python a texto JSON legible |
| `json.loads(texto)` | Convierte texto JSON de vuelta a un diccionario de Python |

Guía de referencia: [`04-despliegue-kserve.md`](04-despliegue-kserve.md). Requiere haber corrido antes [Fundamento 3](03-entrenamiento-modelo.ipynb).


## 0. Preparativos: cargar el modelo entrenado

Aquí ocurre algo conceptualmente importante: **cargamos un modelo sin volver a entrenarlo**.

1. **Imports**: `joblib` para recuperar el modelo, `pandas` para armar la fila de entrada, y `json` + `urllib` para la parte HTTP del final. Fíjate en que se usa `urllib` (librería estándar) en vez de `requests`: una dependencia menos que instalar en la imagen del contenedor.
2. **`assert model_path.exists()`**: si no corriste el Fundamento 3, la celda falla aquí con un mensaje claro en vez de dar un error críptico más abajo.
3. **`joblib.load(...)`**: deserializa el archivo y reconstruye en memoria el objeto de Python exacto que se guardó — preprocesador **y** clasificador con sus pesos. Es la operación inversa a `joblib.dump()`.

**Por qué esto es el corazón del serving:** en la API real esta carga ocurre **una sola vez, al arrancar el proceso**, y el objeto queda en memoria. Cada petición reutiliza ese mismo objeto; leer el `.pkl` del disco en cada request multiplicaría la latencia sin ninguna ventaja. Es el mismo patrón que un pool de conexiones a base de datos.

**Qué mirar en la salida:** la ruta absoluta del `model.pkl` cargado. Si salta el `AssertionError`, ejecuta antes [03-entrenamiento-modelo.ipynb](03-entrenamiento-modelo.ipynb).

> Detalle de seguridad que conviene mencionar en clase: deserializar un `.pkl` **ejecuta código**. Nunca cargues un pickle de origen desconocido; en producción, los modelos deben venir de un registro de modelos con control de acceso y trazabilidad.


In [ ]:
import json  # construir/leer el payload y la respuesta en formato JSON
import urllib.request  # hacer la petición HTTP POST a la API (sin depender de librerías externas)
import urllib.error  # capturar errores de conexión al llamar a la API
from pathlib import Path  # rutas de archivo multiplataforma
import joblib  # cargar el pipeline entrenado (model.pkl)
import pandas as pd  # armar el DataFrame de entrada para el modelo

ARTIFACT_DIR = Path("artifacts")  # misma carpeta donde el Fundamento 3 guardó el modelo
model_path = ARTIFACT_DIR / "model.pkl"
assert model_path.exists(), "Corre primero el notebook 03-entrenamiento-modelo.ipynb"  # falla temprano con un mensaje claro

pipeline = joblib.load(model_path)  # deserializa el pipeline (preprocesador + clasificador) entrenado
print(f"Pipeline cargado desde: {model_path.resolve()}")

## El contrato de la API

Un **contrato** aquí significa: qué campos entran, con qué nombres, en qué orden y con qué reglas se derivan los que faltan. La celda siguiente define ese contrato, replicando lo que hacen `inference/src/schemas.py` y `inference/src/predictor.py`.

### El problema de las 15 vs. 19

| Origen | Cuántos campos | Por qué |
|---|---|---|
| Formulario del frontend | **15** | Solo se le pide a la persona lo que puede saber y teclear |
| Lo que el modelo espera | **19** | Incluye las 4 features derivadas que se crearon en el Fundamento 2 |

La función `to_model_input()` es exactamente ese puente: recibe los 15 campos crudos y calcula las 4 que faltan (`RoleStagnationRatio`, `TenureGap`, `EarlyCompanyTenureRisk`, `LongTenureLowRoleRisk`). **Las fórmulas deben ser idénticas a las del Fundamento 2.** Si aquí se calculara `TenureGap` de otra manera, el modelo recibiría un dato con un significado distinto al que aprendió: es el clásico *training/serving skew*, el fallo más difícil de detectar en producción porque **no lanza ningún error**.

### Por qué existe `FEATURE_ORDER`

Cuando llega el momento de predecir, el modelo ya no ve nombres de columna: ve **posiciones**. Si `Overtime` llega en la posición donde entrenó `Job Level`, la predicción se calcula igual de rápido… y está mal, en silencio.

> Regla para llevarse a producción: **el orden de las features es parte del contrato del modelo**, igual que el esquema de una API REST. Debe versionarse junto al `.pkl`.

También se define `THRESHOLD = 0.50`: la probabilidad a partir de la cual se decide "se va". No es un número técnico, es una **decisión de negocio** (bajarlo detecta a más gente en riesgo, a costa de más falsas alarmas).

**Qué mirar en la salida:** nada, la celda solo define constantes y una función. El trabajo se ve en la celda siguiente.


In [ ]:
# Orden exacto de columnas que el modelo espera recibir (debe coincidir con X_train del Fundamento 3)
FEATURE_ORDER = [
    "Years at Company", "Performance Rating", "Number of Promotions",
    "Overtime", "Education Level", "Number of Dependents",
    "Job Level", "Company Size", "Company Tenure", "Remote Work",
    "Company Reputation", "OverallSatisfaction", "Opportunities",
    "AnnualIncome", "AgeGroup", "RoleStagnationRatio", "TenureGap",
    "EarlyCompanyTenureRisk", "LongTenureLowRoleRisk",
]
THRESHOLD = 0.50  # a partir de qué probabilidad de "se va" (p_leave) se predice que el empleado se irá


def to_model_input(raw: dict) -> dict:
    # Replica EmployeeFeatures.to_model_input(): calcula las 4 features derivadas
    yac, ct, jl = raw["years_at_company"], raw["company_tenure"], raw["job_level"]  # variables cortas usadas en varias fórmulas
    return {
        "Years at Company": yac,
        "Performance Rating": raw["performance_rating"],
        "Number of Promotions": raw["no_of_promotions"],
        "Overtime": raw["overtime"],
        "Education Level": raw["edu_level"],
        "Number of Dependents": raw["no_of_dependents"],
        "Job Level": jl,
        "Company Size": raw["company_size"],
        "Company Tenure": ct,
        "Remote Work": raw["remote_work"],
        "Company Reputation": raw["company_reputation"],
        "OverallSatisfaction": raw["overall_satisfaction"],
        "Opportunities": raw["opportunities"],
        "AnnualIncome": raw["annual_income"],
        "AgeGroup": raw["age_group"],
        "RoleStagnationRatio": round(yac / (ct + 1), 3),  # misma fórmula del feature engineering (Fundamento 2); +1 evita división entre cero
        "TenureGap": round(ct - yac, 2),  # diferencia entre antigüedad total y tiempo en el rol actual
        "EarlyCompanyTenureRisk": 1 if yac <= 2 else 0,  # 1 si lleva 2 años o menos en la empresa
        "LongTenureLowRoleRisk": 1 if (ct > 5 and jl <= 2) else 0,  # 1 si lleva mucho tiempo sin ascender
    }

## Simular lo que hace `predictor.py` al recibir un request

Esta celda recorre **una petición completa**, exactamente en el mismo orden en que la procesaría la API real. Es la celda clave del notebook:

| Paso del código | Qué representa en la API real |
|---|---|
| `raw_request = {...}` | El JSON que envía el formulario del frontend (nombres en `snake_case`) |
| `to_model_input(raw_request)` | La validación y el enriquecimiento de `schemas.py`: 15 campos → 19 features |
| `pd.DataFrame([...])[FEATURE_ORDER]` | Armar la fila en el orden exacto que espera el modelo |
| `pipeline.predict_proba(row)[0]` | La inferencia propiamente dicha |
| `response = {...}` | El JSON que la API devuelve al frontend |

**Por qué `predict_proba()` y no `predict()`:** `predict()` devolvería un 0 o un 1 y se acabó. `predict_proba()` devuelve las dos probabilidades (suman 1), y con ellas se puede hacer algo mucho más útil para RRHH: **priorizar**. No es lo mismo un 0.51 que un 0.94.

**Los niveles de riesgo** traducen esa probabilidad a un lenguaje accionable para el negocio:

| `p_leave` | Riesgo | Lectura para RRHH |
|---|---|---|
| ≥ 0.65 | `HIGH` | Actuar esta semana |
| ≥ 0.45 | `MEDIUM` | Programar una conversación |
| ≥ 0.25 | `LOW` | Vigilar |
| < 0.25 | `VERY_LOW` | Sin acción |

**Detalles técnicos que suelen preguntarse:**

- **`int(p_leave >= THRESHOLD)`**: en Python, `True`/`False` convertidos a entero dan `1`/`0`. Así se obtiene la predicción binaria a partir del umbral.
- **`float(...)` y `round(..., 4)`**: `predict_proba` devuelve tipos de numpy, que el serializador JSON no sabe convertir. Pasarlos a `float` nativo evita un error al construir la respuesta — un detalle pequeño que rompe APIs reales con frecuencia.
- **`json.dumps(response, indent=2)`**: se imprime como JSON, no como diccionario de Python, para ver **literalmente** lo que viajaría por la red.

**Qué mirar en la salida:** el JSON completo. Ese es el contrato de respuesta que el frontend espera; cualquier cambio de nombres o de tipos ahí rompe la interfaz.

**Experimento recomendado:** cambia `overtime` a `0` o sube `overall_satisfaction` en `raw_request`, vuelve a ejecutar y observa cómo se mueven `p_leave` y el nivel de riesgo. Es la misma demostración que harás con el formulario web al final del fundamento.


In [ ]:
# Payload "crudo": exactamente lo que enviaría el formulario del frontend, con nombres de campo en snake_case
raw_request = {
    "years_at_company": 0.5, "performance_rating": 2, "no_of_promotions": 0,
    "overtime": 1, "edu_level": 2, "no_of_dependents": 1, "job_level": 1,
    "company_size": 2, "company_tenure": 1.0, "remote_work": 0,
    "company_reputation": 2, "overall_satisfaction": 1, "opportunities": 0,
    "annual_income": 1, "age_group": 2,
}

model_input = to_model_input(raw_request)  # calcula las 4 features derivadas y arma el diccionario completo
row = pd.DataFrame([model_input])[FEATURE_ORDER]  # DataFrame de 1 fila, columnas reordenadas al orden que espera el modelo

p_stay, p_leave = pipeline.predict_proba(row)[0]  # probabilidad de cada clase: [P(se queda), P(se va)]
response = {
    "prediction": int(p_leave >= THRESHOLD),  # 1 si supera el umbral de decisión, 0 si no
    "p_leave": round(float(p_leave), 4),
    "p_stay": round(float(p_stay), 4),
    "risk": "HIGH" if p_leave >= 0.65 else "MEDIUM" if p_leave >= 0.45 else "LOW" if p_leave >= 0.25 else "VERY_LOW",  # categoría de riesgo para RRHH
    "threshold": THRESHOLD,
}
print(json.dumps(response, indent=2))  # respuesta con el mismo formato que devolvería la API real

## Probar la API real, si ya está corriendo (opcional)

Hasta aquí llamamos al modelo **dentro del propio notebook**. Esta celda hace lo mismo, pero **por HTTP**: es la diferencia entre "tengo un modelo" y "tengo un servicio".

Si tienes la API levantada localmente:

```bash
cd ../02-phase-1-local-dev-mlops/inference
uvicorn src.app:app --host 0.0.0.0 --port 8080
```

…o expuesta con `kubectl port-forward` desde KServe, esta celda le hará un `POST` real a `/predict`. Si no está disponible, imprime cómo levantarla y no rompe el notebook.

**Cómo se construye la petición, pieza por pieza:**

| Pieza | Para qué sirve |
|---|---|
| `json.dumps(raw_request).encode("utf-8")` | El cuerpo de la petición: el diccionario pasa a texto JSON y luego a bytes |
| `headers={"Content-Type": "application/json"}` | Le dice al servidor cómo interpretar ese cuerpo |
| `method="POST"` | Se envían datos, no se piden (por eso no es `GET`) |
| `timeout=3` | No esperar indefinidamente si el servicio no responde |
| `json.loads(resp.read())` | Deshacer el camino: bytes → texto JSON → diccionario |

**Fíjate en un detalle clave:** se envía **el mismo `raw_request`** de la celda anterior, los 15 campos crudos. El cálculo de las 4 features derivadas ocurre **dentro** de la API. Si la respuesta HTTP coincide con la que calculamos localmente, el contrato está bien implementado — y esa comparación es, en esencia, un test de integración del modelo.

**Qué mirar en la salida:** o bien el JSON de la API (idéntico en estructura al de la celda anterior), o bien el mensaje de "no se pudo conectar" con las instrucciones para levantarla. Que falle no significa que algo esté mal: la celda es opcional.

> **Puente hacia Kubernetes:** esa misma API, metida en una imagen Docker y declarada como `InferenceService`, es lo que KServe despliega. KServe además consulta `/health` y `/ready` para saber cuándo el pod está listo para recibir tráfico, y añade autoescalado y gestión de tráfico. Los pasos completos están en [`04-despliegue-kserve.md`](04-despliegue-kserve.md).


In [ ]:
API_URL = "http://localhost:8080/predict"  # cambia el host/puerto si corres la API distinto

try:
    req = urllib.request.Request(
        API_URL,
        data=json.dumps(raw_request).encode("utf-8"),  # el mismo payload crudo, codificado a bytes UTF-8 para el body HTTP
        headers={"Content-Type": "application/json"},  # le dice a la API que el body es JSON
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=3) as resp:  # envía la petición; espera máx. 3 segundos por respuesta
        print("Respuesta de la API:")
        print(json.dumps(json.loads(resp.read()), indent=2))  # lee bytes -> dict (json.loads) -> texto legible (json.dumps)
except (urllib.error.URLError, ConnectionError, TimeoutError) as e:  # la API no está corriendo o no responde a tiempo
    print(f"No se pudo conectar a {API_URL} ({e}).")
    print("Levanta la API primero, por ejemplo:")
    print("  cd ../02-phase-1-local-dev-mlops/inference && uvicorn src.app:app --host 0.0.0.0 --port 8080")

## Ideas clave

- El trabajo del Data Scientist termina en validar el contrato de entrada/salida del modelo; empaquetarlo en Docker/Kubernetes es trabajo de MLOps/Infraestructura.
- El orden de `FEATURE_ORDER` debe coincidir exactamente con el orden usado durante el entrenamiento — un desorden no genera error, solo predicciones incorrectas silenciosas.
- Las features derivadas deben calcularse con las **mismas fórmulas** en entrenamiento y en inferencia (esto es lo que resuelve un Feature Store).
- El modelo se carga una sola vez al arrancar el servicio, no en cada petición.
- `/health` y `/ready` no son opcionales: KServe los usa para gestionar el ciclo de vida del pod.
- Con esto se cierra la Fase 1: dataset → features → modelo entrenado → contrato de API validado, listo para desplegarse.

## Si te perdiste, quédate con esto

Todo el notebook responde a una sola pregunta: **¿qué tiene que pasar entre el formulario y la respuesta?**

```text
15 campos del formulario
   → +4 features derivadas (mismas fórmulas del Fundamento 2)
   → reordenar según FEATURE_ORDER
   → predict_proba()
   → JSON: prediction + p_leave + p_stay + risk
```

| Riesgo del despliegue | Cómo se evita |
|---|---|
| Features calculadas distinto que en entrenamiento | Reutilizar las mismas fórmulas (o un Feature Store) |
| Columnas en otro orden | Reordenar siempre con `FEATURE_ORDER` |
| Recargar el modelo en cada request | Cargarlo una vez al arrancar y mantenerlo en memoria |
| Kubernetes manda tráfico a un pod que aún no cargó el modelo | Endpoints `/health` y `/ready` |

## Siguiente paso

El despliegue real (Docker → cert-manager → KServe → `InferenceService` → frontend) está en la guía [`04-despliegue-kserve.md`](04-despliegue-kserve.md). Con eso se cierra la Fase 1 y comienza la [Fase 2](../04-phase-2-enterprise-setup-mlops/README.md), donde todo esto se automatiza, versiona y monitoriza.
